# Data Cleaning — Customer Dataset

### Cleaning Steps
1. Load raw CSV
2. Inspect shape, dtypes, missing values
3. Fix data types
4. Remove duplicates
5. Handle missing values
6. Fix invalid numeric values
7. Handle outliers
8. Normalize categorical values (casing)
9. Final validation
10. Save cleaned CSV

## Step 1 — Load Raw Data

In [3]:
import pandas as pd
import numpy as np

RAW_PATH = '../../../data/customers_raw.csv'
CLEAN_PATH = '../../../data/customers_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (5600, 15)


,customer_id,age,city,customer_segment,product_category,order_count,average_order_value,total_spend,discount_percentage,days_since_last_order,website_visits,support_tickets,return_count,payment_type,customer_tenure_days
0,1,58.0,Chennai,Premium,sports,18.0,12622.15,227198.70,6.98,53,174,17,2,Cash,894
1,2,20.0,Chennai,premium,Clothing,15.0,25515.09,382726.35,1.33,102,184,20,13,Debit Card,949
2,3,55.0,delhi,Premium,Books,11.0,35057.90,385636.90,17.01,80,56,10,1,credit card,808
3,4,24.0,bangalore,REGULAR,furniture,17.0,40452.85,687698.45,36.49,275,32,12,2,net banking,630
4,5,58.0,bangalore,Inactive,Clothing,46.0,3942.98,181377.08,33.06,149,21,7,6,upi,599


## Step 2 — Inspect: Dtypes, Missing Values, Duplicates

In [8]:
print('=== Data Types ===')
print(df.dtypes)

=== Data Types ===
customer_id                int64
age                      float64
city                         str
customer_segment             str
product_category             str
order_count              float64
average_order_value      float64
total_spend              float64
discount_percentage      float64
days_since_last_order      int64
website_visits             int64
support_tickets            int64
return_count               int64
payment_type                 str
customer_tenure_days       int64
dtype: object


In [6]:
print('=== Missing Values ===')
print(df.isnull().sum())

=== Missing Values ===
customer_id                0
age                      199
city                     100
customer_segment           0
product_category           0
order_count               40
average_order_value      100
total_spend                0
discount_percentage        0
days_since_last_order      0
website_visits             0
support_tickets            0
return_count               0
payment_type               0
customer_tenure_days       0
dtype: int64


In [7]:
print('=== Duplicate Rows ===')
print(f'Exact duplicates: {df.duplicated().sum()}')

=== Duplicate Rows ===
Exact duplicates: 95


In [9]:
print('=== Basic Statistics ===')
df.describe()

=== Basic Statistics ===


,customer_id,age,order_count,average_order_value,total_spend,discount_percentage,days_since_last_order,website_visits,support_tickets,return_count,customer_tenure_days
count,5600.000000,5401.000000,5560.000000,5500.000000,5.600000e+03,5600.000000,5600.000000,5600.000000,5600.000000,5600.000000,5600.000000
mean,2748.137143,44.134975,25.704137,25367.849751,6.751479e+05,25.532175,183.184821,123.759286,10.162679,12.872857,918.514107
std,1587.252940,15.549018,14.583825,14320.324338,6.548652e+05,16.516657,106.159221,441.652640,6.048958,11.455790,510.955382
min,1.000000,-5.000000,1.000000,504.600000,5.605200e+02,0.010000,1.000000,1.000000,0.000000,0.000000,30.000000
25%,1374.750000,31.000000,13.000000,12859.995000,1.835391e+05,12.907500,91.750000,51.000000,5.000000,3.000000,480.000000
50%,2745.500000,44.000000,26.000000,25537.175000,5.023962e+05,24.860000,184.000000,101.000000,10.000000,10.000000,907.000000
75%,4122.250000,58.000000,39.000000,37746.795000,1.014557e+06,37.360000,275.000000,151.000000,15.000000,20.000000,1362.000000
max,5500.000000,70.000000,50.000000,49985.470000,6.900299e+06,150.000000,365.000000,8573.000000,20.000000,50.000000,1825.000000


## Step 3 — Fix Data Types

**What happened:** `order_count` was written as a string in the generation script with `'N/A'` values injected.
When pandas reads the CSV, it automatically parses the numeric strings and silently converts `'N/A'` to `NaN`, so the column loads as `float64` — not `object`.

The `NaN` values are still present (40 of them) and will be handled in Step 5.
After filling, we cast to `int` since order count is a whole number.

In [ ]:
print('order_count dtype:', df['order_count'].dtype)
print('NaNs in order_count:', df['order_count'].isna().sum())

## Step 4 — Remove Duplicates

**Problem:** 100 exact duplicate rows were injected.

**Strategy:** Drop exact duplicate rows — rows where every column value is identical. We keep the first occurrence.

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Rows before: {before}')
print(f'Rows after:  {after}')
print(f'Duplicates removed: {before - after}')

## Step 5 — Handle Missing Values

| Column | Type | Strategy | Reason |
|--------|------|----------|--------|
| `age` | Numeric | Median | Median is robust to the skew caused by our invalid (-5) values and outliers. Mean would be pulled by extremes. |
| `city` | Categorical | `'Unknown'` | Mode could unfairly over-represent one city. 'Unknown' is honest — we simply don't know. |
| `average_order_value` | Numeric | Median | Same reasoning as age. |
| `order_count` | Numeric | Median | NaNs here came from invalid string 'N/A' entries; median preserves typical behaviour. |

In [ ]:
print('Missing values BEFORE filling:')
print(df[['age', 'city', 'average_order_value', 'order_count']].isnull().sum())

In [ ]:
# Numeric columns — fill with median (calculated on valid values only)
for col in ['age', 'average_order_value', 'order_count']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'{col}: filled NaN with median = {median_val}')

# order_count is always a whole number — cast to int after NaN fill
# (pandas stored it as float64 only because NaN cannot exist in int columns)
df['order_count'] = df['order_count'].round().astype(int)

# Categorical column — fill with 'Unknown'
df['city'] = df['city'].fillna('Unknown')
print('city: filled NaN with Unknown')

In [ ]:
print('Missing values AFTER filling:')
print(df[['age', 'city', 'average_order_value', 'order_count']].isnull().sum())

## Step 6 — Fix Invalid Numeric Values

**Problem 1:** `age` has values of `-5` — a person cannot have a negative age.

**Fix:** Replace values outside the valid range `[1, 110]` with the median age.

**Problem 2:** `discount_percentage` has values of `150` — a discount cannot exceed 100%.

**Fix:** Clip values to the valid range `[0, 100]`.

In [ ]:
# age — invalid: negative values
invalid_age_mask = (df['age'] < 1) | (df['age'] > 110)
print(f'Invalid age values: {invalid_age_mask.sum()}')
print(df.loc[invalid_age_mask, 'age'].value_counts())

median_age = df.loc[~invalid_age_mask, 'age'].median()
df.loc[invalid_age_mask, 'age'] = median_age
print(f'Replaced invalid age with median: {median_age}')

In [ ]:
# discount_percentage — invalid: > 100
invalid_discount_mask = (df['discount_percentage'] < 0) | (df['discount_percentage'] > 100)
print(f'Invalid discount_percentage values: {invalid_discount_mask.sum()}')

# Clip to valid range
df['discount_percentage'] = df['discount_percentage'].clip(lower=0, upper=100)
print('Clipped discount_percentage to [0, 100]')
print(f'Max after clip: {df["discount_percentage"].max()}')

## Step 7 — Handle Outliers

**Method used: IQR (Interquartile Range)**

For a value `x`:
- Lower fence = Q1 − 3×IQR
- Upper fence = Q3 + 3×IQR

We use 3×IQR (extreme outliers only) instead of the standard 1.5×IQR to avoid being too aggressive — legitimate high-value customers should not be removed, only impossible values.

**Columns:** `total_spend`, `website_visits`

**Strategy:** Cap (Winsorize) — values beyond the fence are replaced with the fence value, not deleted. This preserves the record while neutralising the extreme value.

In [ ]:
def cap_outliers_iqr(df: pd.DataFrame, col: str, multiplier: float = 3.0) -> pd.DataFrame:
    """Cap outliers using the IQR method (Winsorization)."""
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Fences: [{lower:.2f}, {upper:.2f}]')
    print(f'  Outliers found: {outliers}')
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'  Capped to fences.')
    return df

df = cap_outliers_iqr(df, 'total_spend')
print()
df = cap_outliers_iqr(df, 'website_visits')

## Step 8 — Normalize Categorical Values

**Problem:** Columns like `city`, `customer_segment`, `product_category`, and `payment_type` have inconsistent casing and leading/trailing whitespace.

Examples:
- `'Chennai'`, `'chennai'`, `'CHENNAI'`, `' Chennai '` → should all be `'Chennai'`

**Fix:** Strip whitespace + title-case for city/segment/category, upper-case for short codes.

In [ ]:
# Before normalization — show unique values
for col in ['city', 'customer_segment', 'product_category', 'payment_type']:
    print(f'\n{col} unique values ({df[col].nunique()}):')
    print(sorted(df[col].dropna().unique()))

In [ ]:
# Normalize: strip whitespace + title case
for col in ['city', 'customer_segment', 'product_category', 'payment_type']:
    df[col] = df[col].str.strip().str.title()

# After normalization
for col in ['city', 'customer_segment', 'product_category', 'payment_type']:
    print(f'\n{col} unique values AFTER ({df[col].nunique()}):')
    print(sorted(df[col].dropna().unique()))

## Step 9 — Final Validation

Confirm the cleaned dataset is free of all the problems we identified.

In [ ]:
print('=== Final Shape ===')
print(df.shape)

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicates ===')
print(f'Remaining duplicates: {df.duplicated().sum()}')

print('\n=== Numeric Range Checks ===')
print(f'age min/max:                  {df["age"].min()} / {df["age"].max()}')
print(f'discount_percentage min/max:  {df["discount_percentage"].min()} / {df["discount_percentage"].max()}')
print(f'order_count min/max:          {df["order_count"].min()} / {df["order_count"].max()}')
print(f'return_count <= order_count:  {(df["return_count"] <= df["order_count"]).all()}')

In [ ]:
print('=== Final Statistics ===')
df.describe()

## Step 10 — Save Cleaned Dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f'Cleaned dataset saved to: {CLEAN_PATH}')
print(f'Final record count: {len(df)}')